In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plot

%matplotlib inline

In [ ]:
def f(x):
    return 3 * x**2 - 4 * x + 5

In [ ]:
f(3.0)

In [ ]:
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plot.plot(xs, ys)
plot.show()

In [ ]:
h = 0.000001
x = 3
(f(x + h) - f(x)) / h

In [ ]:
h = 0.000001
x = -3
(f(x + h) - f(x)) / h

In [ ]:
h = 0.00001
x = 2 / 3
(f(x + h) - f(x)) / h

In [ ]:
a = 2.0
b = -3.0
c = 10.0

h = 0.001
d1 = a * b + c
print(d1)
d2 = (a + h) * b + c

# Rate of change of a => dd/da = b
(d2 - d1) / h

In [ ]:
class Value:
    def __init__(self, data, _children=(), op="", label="") -> None:
        self.data = data
        self._prev = set(_children)  # What all objects are involved to form this Object
        self._op = op  # What operation was performed to get this data
        self.label = label
        self.grad = 0

    def __repr__(self) -> str:
        return f"Value(data={self.data})"

    def __add__(self, other):
        operation = "+"
        self.__type_validate(other, operation)
        return Value(self.data + other.data, (self, other), operation)

    def __sub__(self, other):
        operation = "-"
        self.__type_validate(other, operation)
        return Value(self.data - other.data, (self, other), operation)

    def __mul__(self, other):
        operation = "*"
        self.__type_validate(other, operation)
        return Value(self.data * other.data, (self, other), operation)

    def __truediv__(self, other):
        operation = "/"
        self.__type_validate(other, operation)
        return Value(self.data / other.data, (self, other), operation)

    @staticmethod
    def __type_validate(other, op):
        if not isinstance(other, Value):
            raise TypeError(
                f"unsupported operand type(s) for {op}: 'Value' and '{type(other)}'"
            )

In [ ]:
from graphviz import Digraph


def trace(root):
    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)

    build(root)
    return nodes, edges


def draw_dot(root):
    dot = Digraph(format="svg", graph_attr={"rankdir": "LR"})
    nodes, edges = trace(root)

    for n in nodes:
        uid = str(id(n))

        dot.node(
            name=uid,
            label=f"{{ {n.label} | data: {n.data:.4f} | grad: {n.grad:.4f} }}",
            shape="record",
        )
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [ ]:
a = Value(2, label="a")
b = Value(-3, label="b")

c = Value(10, label="c")
e = a * b
e.label = "e"

d = e + c
d.label = "d"

f = Value(-2, label="f")

L = d * f
L.label = "L"
draw_dot(L)

In [ ]:
plot.plot(np.arange(-5, 5, 0.2), np.tanh(np.arange(-5, 5, 0.2)))
plot.grid()
